# Before the journey begins: what exactly is a token?
### The Hitchhiker's Guide to the Tokenverse, part 1

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srishtiii28/hitchhiker_tokens/blob/main/notebooks/01_understanding_tokenization.ipynb)

> "Time is an illusion. Lunchtime doubly so."
>
> Douglas Adams, *The Hitchhiker's Guide to the Galaxy*

You and I can read that sentence without thinking about it. A language model can't. Before it does anything with those words, it has to turn them into a sequence of tokens, each one an integer ID that is then mapped to a learned vector.

This notebook follows that first conversion. We'll try characters, then whole words, then subwords, build a small BPE tokenizer by hand, and finally put the quote through three real tokenizers to see how differently they cut it up.

Everything up to section 8 runs offline. Section 8 downloads tokenizer files (a few MB, no model weights).

## 0. Setup

In [ ]:
# Setup: run this cell first.
# In Colab it clones the repository and installs the package. Locally it assumes
# you ran `pip install -e ".[nlp,notebooks,vision,dev]"` from the repo root.
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/srishtiii28/hitchhiker_tokens.git"

if IN_COLAB:
    repo_dir = Path("/content/hitchhiker_tokens")
    if not repo_dir.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)])
    %pip install -q -e "/content/hitchhiker_tokens[nlp,notebooks]"
    # Colab ships with PyTorch; install it only if it is missing.
    if importlib.util.find_spec("torch") is None:
        %pip install -q torch
else:
    repo_dir = Path.cwd()
    while not (repo_dir / "pyproject.toml").exists() and repo_dir != repo_dir.parent:
        repo_dir = repo_dir.parent

os.chdir(repo_dir)
# Make the package importable without restarting the runtime after install.
if str(repo_dir / "src") not in sys.path:
    sys.path.insert(0, str(repo_dir / "src"))
print("Working directory:", Path.cwd())

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import torch

from tokenverse.language.toy_bpe import ToyBPE
from tokenverse.visualization import apply_dark_theme

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
apply_dark_theme()
%matplotlib inline

QUOTE = "Time is an illusion. Lunchtime doubly so."

## 1. What does the model actually receive?

Neural networks operate on numbers, so the text has to be converted first. The journey looks like this:

**Text → Tokens → Token IDs → Embeddings → Transformer**

The tokenizer splits the text into pieces from its vocabulary. Each piece is mapped to an integer ID, and the model uses those IDs to pick rows out of its embedding matrix. For GPT-2, the first half of our quote comes out as:

```
"Time is an illusion."  →  ["Time", " is", " an", " illusion", "."]  →  [7575, 318, 281, 17878, 13]
```

(We'll check those numbers against the real tokenizer in section 9.) The interesting question is how the pieces get chosen. There are three obvious answers.

## 2. Character-level tokenisation

The simplest option is to make every character a token. The vocabulary is tiny, and nothing is ever out of vocabulary as long as we include every character we might meet.

In [ ]:
char_tokens = list(QUOTE)
char_vocab = {ch: i for i, ch in enumerate(sorted(set(char_tokens)))}
char_ids = [char_vocab[ch] for ch in char_tokens]

print("tokens:", char_tokens)
print("ids:   ", char_ids)
print(
    f"{len(QUOTE)} characters -> {len(char_ids)} tokens, vocabulary of {len(char_vocab)}"
)

The catch is length. Our 41-character quote becomes 41 tokens, and a 1,000-word essay would be somewhere around 5,000 to 6,000. Self-attention gets more expensive as sequences grow, and the model has to learn from scratch that `L`, `u`, `n`, `c`, `h` together mean something.

## 3. Word-level tokenisation and unfamiliar words

At the other extreme, every word becomes a token. Sequences are short, but the vocabulary now has to contain every word the model will ever see. Anything missing becomes an unknown token.

In [ ]:
training_text = "time is an illusion . lunch is lunch . time flies ."
word_vocab = {"[UNK]": 0}
for word in training_text.split():
    word_vocab.setdefault(word, len(word_vocab))


def word_encode(sentence):
    return [word_vocab.get(w, word_vocab["[UNK]"]) for w in sentence.split()]


for sentence in ["time is an illusion .", "lunchtime doubly so ."]:
    ids = word_encode(sentence)
    shown = [w if word_vocab.get(w) else "[UNK]" for w in sentence.split()]
    print(f"{sentence!r:28} -> {ids}  {shown}")

`lunchtime`, `doubly` and `so` were never seen, so they all collapse into the same `[UNK]` ID and the model can't tell them apart. It also has no way of knowing that `lunchtime` has anything to do with `lunch`. A bigger vocabulary helps, but there will always be names, typos, numbers, code identifiers and words from other languages that it hasn't seen.

## 4. Subwords: the practical middle ground

Subword tokenizers keep common words whole and build rarer ones out of smaller, reusable pieces. `lunchtime` can become `lunch` + `time`, and a completely unfamiliar word falls back to short fragments or, at worst, single characters or bytes. The vocabulary stays at a manageable size (tens of thousands of entries) and nothing is ever truly unknown.

The exact boundaries depend on the algorithm, the training data and the vocabulary the tokenizer learns. The most widely used algorithm is Byte-Pair Encoding.

## 5. Token strings, token IDs and embeddings

These three get blurred together quite easily:

| | Example | What it is |
|---|---|---|
| token string | `" illusion"` | an entry in the vocabulary |
| token ID | `17878` | that entry's index |
| embedding | a vector of, say, 768 numbers | a learned row of the embedding matrix |

The ID is only an address. There's nothing meaningful about 17878 being bigger than 318, and neighbouring IDs aren't related words. Here's a tiny embedding table to make that concrete.

In [ ]:
embedding = torch.nn.Embedding(num_embeddings=len(char_vocab), embedding_dim=4)
ids = torch.tensor(char_ids[:4])  # "Time"
vectors = embedding(ids)
print("ids:", ids.tolist())
print("vectors shape:", tuple(vectors.shape))  # [4 tokens, 4 dims]
print(vectors.detach().numpy().round(3))

The lookup just selects rows: `embedding(ids)[k]` is `embedding.weight[ids[k]]`. These vectors are random because nothing has been trained yet. If we renumbered the whole vocabulary, the IDs would change but a model trained on them would be just as good, because the numbers themselves carry no meaning.

In [ ]:
assert torch.equal(vectors[2], embedding.weight[ids[2]])
print("Same row: an ID is only an index into the table.")

## 6. BPE by hand

BPE was adapted for text by Sennrich, Haddow and Birch in 2016. It builds a vocabulary from the bottom up:

1. Write every word as single characters, plus an end-of-word marker `</w>`.
2. Count every adjacent pair of symbols across the corpus.
3. Merge the most frequent pair into a new symbol everywhere it appears.
4. Repeat until you've done as many merges as you want.

Let's use the same tiny corpus as the article: `low`, `lower`, `lowest`, `low`. At the start, `l o` and `o w` both appear 4 times. When counts tie, `ToyBPE` picks the alphabetically smaller pair, so `l o` merges first, then `lo w`.

## 7. Training `ToyBPE`

`ToyBPE` (in `src/tokenverse/language/toy_bpe.py`) follows exactly those steps and records each merge. It's a simplified version: it works on characters, splits on whitespace and has none of GPT-2's byte-level alphabet, regex pre-tokenisation or special tokens.

In [ ]:
bpe = ToyBPE().train("low lower lowest low", num_merges=6)

for step in bpe.merge_history:
    a, b = step.pair
    print(
        f"merge {step.step}: {a!r:>6} + {b!r:<7} (seen {step.frequency}x)"
        f"  lowest -> {step.samples['lowest']}"
    )

In [ ]:
from tokenverse.visualization.text_visualization import plot_bpe_merges

plot_bpe_merges(bpe, ["low", "lower", "lowest"])
plt.show()

Now let's apply those merges to words the trainer never saw. Characters it doesn't know simply stay as single symbols.

In [ ]:
for sample in ["slow", "flower", "lowly", "naïve"]:
    tokens = bpe.encode(sample)
    print(f"{sample!r:10} -> {tokens}")

`slow` reuses `low</w>`, and `flower` reuses the whole of `lower`. `naïve` shares nothing with the corpus, so it stays as single characters. That's the subword idea in miniature: frequent pieces become units, and everything else is assembled from smaller ones.

One caveat: `ToyBPE` collapses repeated spaces and tabs, so `decode` gives back the words but not necessarily the exact whitespace.

## 8. Three real tokenizers

The tokenizers used in practice differ in details that matter:

- **GPT-2 uses byte-level BPE.** It runs BPE over UTF-8 bytes rather than characters, so any string can be encoded without an unknown token. Bytes are mapped to printable characters, which is why a leading space appears as `Ġ` in GPT-2's token strings.
- **BERT (`bert-base-uncased`) uses WordPiece.** It lowercases the text and strips accents first, then splits each word into the longest pieces it knows. A piece that continues a word gets a `##` prefix.
- **T5 (`t5-small`) uses SentencePiece with a Unigram model.** Spaces are marked with `▁`. The model gives each piece a probability and picks the most probable way to split each word.

The next cell downloads only the tokenizer files. If you're offline, it prints the error and falls back to a tiny tokenizer trained locally, so the rest of the notebook still runs.

In [ ]:
from tokenverse.language.local_tokenizer import build_local_tokenizer
from tokenverse.language.tokenizer_loader import (
    SUPPORTED_TOKENIZERS,
    load_supported_tokenizers,
)

tokenizers, errors = load_supported_tokenizers()
for name, info in SUPPORTED_TOKENIZERS.items():
    status = "loaded" if name in tokenizers else "FAILED"
    print(f"{name:<18} {info.family:<22} {status}")
for name, message in errors.items():
    print(f"\n{name}: {message}")
if not tokenizers:
    print("\nNo pretrained tokenizer available, so using the tiny local BPE instead.")
    tokenizers = {"local-tiny-bpe": build_local_tokenizer()}

## 9. Our quote, inspected

`inspect_text` returns the token strings, their IDs, their offsets (which characters of the input each token covers) and the decoded text. Spaces in the labels are drawn as `␣` so you can see them.

In [ ]:
from tokenverse.language.tokenizer_inspector import inspect_text
from tokenverse.visualization.text_visualization import plot_token_overlay

first = next(iter(tokenizers))
report = inspect_text(QUOTE, tokenizers[first])
print(report.format_table())
print(f"\ndecoded: {report.decoded!r}   exact round trip: {report.round_trip}")
plot_token_overlay(report)
plt.show()

The `token` column is the vocabulary entry, with GPT-2's `Ġ` marking a leading space. The `label` column is the matching slice of the original text. Notice `doubly`: GPT-2 doesn't have it as a single token, so it becomes ` doub` + `ly`. The IDs only mean something inside GPT-2's vocabulary.

## 10. The same text, three tokenizers

`compare_tokenizers` runs one string through each tokenizer and shows the counts and pieces side by side. It doesn't line up the IDs, because GPT-2's ID 995 and BERT's ID 995 have nothing to do with each other.

In [ ]:
from tokenverse.language.tokenizer_inspector import EXAMPLES, compare_tokenizers
from tokenverse.visualization.text_visualization import (
    plot_token_counts,
    plot_tokenizer_rows,
)

for text in EXAMPLES:
    print(compare_tokenizers(text, tokenizers).summary(), end="\n\n")

Each tokenizer trips over a different word. On the quote, GPT-2 gives ` doub | ly`, BERT gives `do | ub | ly` and T5 gives `doubl | y`. On `Broken bottles break.` it's GPT-2's turn to split a common word, into `Bro | ken`. None of these is wrong. Each tokenizer has a fixed vocabulary learned from its own training data, and which words earned a single token depends on what that data happened to contain.

In [ ]:
plot_tokenizer_rows(compare_tokenizers(QUOTE, tokenizers))
plt.show()
counts = {
    text: compare_tokenizers(text, tokenizers).token_counts() for text in EXAMPLES
}
plot_token_counts(counts)
plt.show()

A few more things you'll probably spot:

- BERT lowercases and strips accents, so `café` comes back as `cafe`. That's why `round_trip` can be `False` for BERT even though nothing has gone wrong.
- GPT-2 can spend several byte tokens on one accented or Chinese character. The extra pieces share the same characters and are labelled `(cont.)`.
- T5 sometimes emits a bare `▁` as its own token (labelled `␣`) before a piece.
- Each tokenizer splits the long digit string differently. Notebook 2 picks up from there.

In [ ]:
for name, tok in tokenizers.items():
    r = inspect_text("café naïve résumé", tok)
    print(f"{name:<18} decoded={r.decoded!r:24} round_trip={r.round_trip}")

## 11. Why any of this matters

The same text costs a different number of tokens under different tokenizers, and that affects how much fits in a context window and how much compute a request takes. It also decides what the model sees as a unit. A word that arrives as one token and the same word split into three are different inputs, and the model has to learn that they mean the same thing.

Tokenisation isn't a universal rule of language processing. It's a decision made by the tokenizer a particular model happens to use.

## 12. Things to try

1. Put your own name, a hashtag or a URL through `compare_tokenizers`. Which tokenizer uses the fewest tokens?
2. Train `ToyBPE` with `num_merges=30` on a paragraph you like. Which merges come first?
3. Find a word that BERT keeps whole but GPT-2 splits, and one where it's the other way round.
4. Try `"👍🏽 ok"`. How many tokens does each tokenizer use, and how many characters does Python count?

Next up, **Notebook 2: the number problem, and when token boundaries get in the way**.